In [1]:
# run cell to install prokbert lib (if necessary)
# !pip install git+https://github.com/nbrg-ppcu/prokbert.git@development --quiet

In [2]:
import os
import hashlib

import datasets
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, balanced_accuracy_score
from transformers import Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput

from prokbert.tokenizer import LCATokenizer
from prokbert.data_collator import SegmentDataCollator
from prokbert.sequence_dataset import SequenceDataset, EmbeddingDataset
from prokbert.dataset import RandomSegmentDataset, ContiguousSegmentDataset

In [3]:
os.environ["WANDB_PROJECT"]="genome-network-pretraining"
os.environ["HF_HOME"] = "/home/datasets/experiments/experiment-bioinfo/cache-for-hugginface"

In [4]:
base_dir = "/home/datasets/experiments/experiment-bioinfo/fine-tuning"

## Dataset

In [33]:
train_ds = datasets.load_dataset("neuralbioinfo/PhaStyle-SequenceDB", split="BACPHLIP_TRAINING")
test_ds  = datasets.load_dataset("neuralbioinfo/PhaStyle-SequenceDB", split="ESCHERICHIA")

In [34]:
print(f"train examples: {len(train_ds)}, columns: {train_ds.column_names}")
print(f"test examples: {len(test_ds)}, columns: {test_ds.column_names}")

train examples: 1798, columns: ['sequence_id', 'dataset', 'class_label', 'y', 'source_fasta_id', 'orientation', 'test_fastaid', 'L_seq', 'source_description', 'Genome type', 'Family', 'subds_seqid', 'sequence', '__index_level_0__']
test examples: 394, columns: ['sequence_id', 'dataset', 'class_label', 'y', 'source_fasta_id', 'orientation', 'test_fastaid', 'L_seq', 'source_description', 'Genome type', 'Family', 'subds_seqid', 'sequence', '__index_level_0__']


In [35]:
# remove cols
cols = [
    "dataset", "class_label", "test_fastaid", "L_seq", "source_description",
    "Genome type", "Family", "subds_seqid", "__index_level_0__"
]
train_ds = train_ds.remove_columns(cols)
test_ds = test_ds.remove_columns(cols)

In [36]:
train_ds = train_ds.rename_column("y", "label")
train_ds = train_ds.rename_column("source_fasta_id", "contig_id")

test_ds = test_ds.rename_column("y", "label")
test_ds = test_ds.rename_column("source_fasta_id", "contig_id")

In [37]:
train_ds = train_ds.filter(lambda x: x["orientation"] == "forward")
test_ds = test_ds.filter(lambda x: x["orientation"] == "forward")

In [41]:
print(train_ds.with_format("pandas")['label'].value_counts())
print(test_ds.with_format("pandas")['label'].value_counts())

label
0    551
1    348
Name: count, dtype: int64
label
0    100
1     97
Name: count, dtype: int64


In [42]:
# check example
# train_ds[0]

In [5]:
train_save_dir = f"{base_dir}/example2-train-dataset/"
test_save_dir = f"{base_dir}/example2-test-dataset/"

In [ ]:
train_ds.save_to_disk(train_save_dir)
test_ds.save_to_disk(test_save_dir)

### Sequence Dataset

In [6]:
train_seq_ds = SequenceDataset()
train_seq_save_dir = f"{base_dir}/example2-train-sequence-dataset"
train_seq_ds.create_dataset([train_save_dir], save_dir=train_seq_save_dir)

print("Total length:", len(train_seq_ds.sequence))
print(f"Total contigs: {len(train_seq_ds.metadata)}")

Total length: 58340084
Total contigs: 899


In [7]:
pd.DataFrame(train_seq_ds.metadata).head()

,genome_id,contig_id,sequence_id,coordinate,orientation,description,label
0,None,PHAA0000323,0,"(0, 53124)",forward,None,0
1,None,PHAA0000761,1,"(53124, 124689)",forward,None,1
2,None,PHAA0000840,2,"(124689, 178085)",forward,None,0
3,None,PHAA0000271,3,"(178085, 249650)",forward,None,1
4,None,PHAA0000060,4,"(249650, 302180)",forward,None,0


In [8]:
test_seq_ds = SequenceDataset()
test_seq_save_dir = f"{base_dir}/example2-test-sequence-dataset"
test_seq_ds.create_dataset([test_save_dir], save_dir=test_seq_save_dir)

print("Total length:", len(test_seq_ds.sequence))
print(f"Total contigs: {len(test_seq_ds.metadata)}")

Total length: 11199891
Total contigs: 197


In [9]:
pd.DataFrame(test_seq_ds.metadata).head()

,genome_id,contig_id,sequence_id,coordinate,orientation,description,label
0,None,409_P1,0,"(0, 40139)",forward,None,1
1,None,409_P3,1,"(40139, 85787)",forward,None,1
2,None,409_P5,2,"(85787, 131435)",forward,None,1
3,None,409_P6,3,"(131435, 176284)",forward,None,1
4,None,409_P8,4,"(176284, 219938)",forward,None,1


In [10]:
# quick check
# for i, meta in enumerate(train_seq_ds.metadata):
#     start, end = meta["coordinate"]
#     print(f"{i:3}: {meta['contig_id']:12} [{start:3}, {end:3})  {train_seq_ds.sequence[start:end]}")
#     if i >= 5: break

### Embedding dataset

Run the embedding script for **train** dataset:
    
```bash
srun --job-name=phastyle-db-test-create-embedding-with-mini2c \
     --nodes=1 \
     --nodelist=bolka \
     --ntasks=4 \
     --gpus-per-task=1 \
     --cpus-per-task=4 \
     --mem=200G \
     --time=7-00:00:00 \
     --partition=hi \
     --output=/home/osvathm/prokbert/logs-embeds2/%x_%j_%t.out \
     --error=/home/osvathm/prokbert/logs-embeds2/%x_%j_%t.err \
     bash -c 'python3 /home/osvathm/prokbert/scripts/embed_dataset.py \
        --model /home/datasets/experiments/experiment-bioinfo/models/mini2-c \
        --input-dir /home/datasets/experiments/experiment-bioinfo/fine-tuning/example2-test-sequence-dataset \
        --output-dir /home/datasets/experiments/experiment-bioinfo/fine-tuning/example2-test-embedding-dataset \
        --seq-len 2000 \
        --pooling-len 100 \
        --batch-size 32 \
        --kmer 1 \
        --shift 1 \
        --pooling-strategy mean \
        --shard-id $SLURM_PROCID \
        --num-shards $SLURM_NTASKS \
        --verbose'

srun --job-name=phastyle-db-test-concat-embedding \
     --nodes=1 \
     --cpus-per-task=4 \
     --mem=150G \
     --partition=hi \
     --output=/home/osvathm/prokbert/logs-embeds2/%x_%j_%t.out \
     --error=/home/osvathm/prokbert/logs-embeds2/%x_%j_%t.err \
     python3 /home/osvathm/prokbert/scripts/embed_dataset.py \
        --model /home/datasets/experiments/experiment-bioinfo/models/mini2-c \
        --input-dir /home/datasets/experiments/experiment-bioinfo/fine-tuning/example2-test-sequence-dataset \
        --output-dir /home/datasets/experiments/experiment-bioinfo/fine-tuning/example2-test-embedding-dataset \
        --seq-len 2000 \
        --pooling-len 100 \
        --kmer 1 \
        --shift 1 \
        --pooling-strategy mean \
        --concat-only
```

Run the embedding script for **train** dataset:

```bash
srun --job-name=phastyle-db-train-create-embedding-with-mini2c \
     --nodes=1 \
     --nodelist=bolka \
     --ntasks=5 \
     --gpus-per-task=1 \
     --cpus-per-task=4 \
     --mem=200G \
     --time=7-00:00:00 \
     --partition=hi \
     --output=/home/osvathm/prokbert/logs-embeds2/%x_%j_%t.out \
     --error=/home/osvathm/prokbert/logs-embeds2/%x_%j_%t.err \
     bash -c 'python3 /home/osvathm/prokbert/scripts/embed_dataset.py \
        --model /home/datasets/experiments/experiment-bioinfo/models/mini2-c \
        --input-dir /home/datasets/experiments/experiment-bioinfo/fine-tuning/example2-train-sequence-dataset \
        --output-dir /home/datasets/experiments/experiment-bioinfo/fine-tuning/example2-train-embedding-dataset \
        --seq-len 2000 \
        --pooling-len 100 \
        --batch-size 32 \
        --kmer 1 \
        --shift 1 \
        --pooling-strategy mean \
        --shard-id $SLURM_PROCID \
        --num-shards $SLURM_NTASKS \
        --verbose'

srun --job-name=phastyle-db-train-concat-embedding \
     --nodes=1 \
     --cpus-per-task=4 \
     --mem=150G \
     --partition=hi \
     --output=/home/osvathm/prokbert/logs-embeds2/%x_%j_%t.out \
     --error=/home/osvathm/prokbert/logs-embeds2/%x_%j_%t.err \
     python3 /home/osvathm/prokbert/scripts/embed_dataset.py \
        --model /home/datasets/experiments/experiment-bioinfo/models/prokbert-mini-long \
        --input-dir /home/datasets/experiments/experiment-bioinfo/fine-tuning/example2-train-sequence-dataset \
        --output-dir /home/datasets/experiments/experiment-bioinfo/fine-tuning/example2-train-embedding-dataset \
        --seq-len 2000 \
        --pooling-len 100 \
        --kmer 1 \
        --shift 1 \
        --pooling-strategy mean \
        --concat-only
```

#### Load dataset

In [17]:
emb_ds_train = EmbeddingDataset(
    sequence_dataset=train_seq_ds,
    config_path=f"{train_seq_save_dir}/config.yaml",
    embedding_file=f"{train_seq_save_dir}/embeddings_forward.pt",
    seq_tokens_file=f"{train_seq_save_dir}/seq_tokens.pt",
)

In [18]:
emb_ds_test = EmbeddingDataset(
    sequence_dataset=test_seq_ds,
    config_path=f"{test_seq_save_dir}/config.yaml",
    embedding_file=f"{test_seq_save_dir}/embeddings_forward.pt",
    seq_tokens_file=f"{test_seq_save_dir}/seq_tokens.pt",
)

In [19]:
print(emb_ds_train.config)
print(emb_ds_test.config)

{'tokenizer': 'LCATokenizer', 'kmer': 6, 'shift': 2, 'model': '/home/datasets/experiments/experiment-bioinfo/models/prokbert-mini-long', 'pooling_strategy': 'mean', 'pooling_length': 100, 'special_tokens': []}
{'tokenizer': 'LCATokenizer', 'kmer': 6, 'shift': 2, 'model': '/home/datasets/experiments/experiment-bioinfo/models/prokbert-mini-long', 'pooling_strategy': 'mean', 'pooling_length': 100, 'special_tokens': []}


## Data Loader and Sampling 

In [20]:
# parameters
min_segment_length = 1000
max_segment_length = 2000

kmer = 6
shift = 2
add_special_tokens = False
return_metadata = False

batch_size = 16

### Sequence Segment Dataset

In [21]:
train_dataset = RandomSegmentDataset(
    sequence_dataset=train_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    return_sequence=True,
    coverage=2,
    seed=42,
)
test_dataset = ContiguousSegmentDataset(
    sequence_dataset=test_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    return_sequence=True,
)

In [22]:
for i, segment in enumerate(iter(train_dataset)):
    if i >= 5: break
    print(segment)

{'segment_id': 52833, 'contig_id': 'PHAA0000796', 'genome_id': None, 'sequence_id': 821, 'absolute_coordinate': (53526916, 53528916), 'relative_coordinate': (1743, 3743), 'orientation': 'forward', 'label': 0, 'sequence': 'TCCCGGCCTGGACGCTCCACTTCAACCTGAACAGCGGCAAGCCGCTCACCGAGCAGCTCGTCAACACGATCAACAAGACCACGAACGACTTCTTCAACACGTTCGAGCAGCTCTTCGGTGTCGACGTGCAGATGACCGTCACCGACGTGCTCAACTTCAAGCTGAAGGTGACCTCCCTCAAGTCGTTCGATGAGGTCGGAGCTGTCACGTTCGCGGTCGACATCACGTCCACGATGGTGAAGAACTTCTTCAACTCCGTGGCTGGTCTGATCGGAGCGGTCAACACGGTCAACGTCGACTTCTACTGGAACCGGACGTACGACGTGGAGTTCACGGGAGCGCTCGCGCTCACCCCGGTCCCGCCCATTACCGCAGACATCAGCAACCTGACCGGCGCATCGAAGTCGGTCAACATCACCGTGAAGCGCCCCGGCAAGACACGGCTCACCCGGTGGGACTTCGTCATCGACGGTACGAAGGCCGTCCTGAAGGTCGAAAGCGAAGAGGCCGATCTGATCTCCGCTCGCACCCAATGGCAACTGGTCTTCCTCGAAGACGGCGAGCCCGCAGGTGGTGAACCGATCGCTCGCGGATTGGTGTGGGTGCAGGAATGAGGCTCAAGGGATACCCAACGGACGGTAGGCCGGCGCTCTCCTATCTGGGGAGCCCGACCGGCTCCATCGTCGGATCTGCTGAGCGTCCCGTGGGGCACATCATCTCGGTCCCCGGTCAGACCGGGCCTCGCGGCCCGCAGGGTGACATCGGACCGCAGGGTGTC

In [23]:
for i, segment in enumerate(iter(test_dataset)):
    if i >= 5: break
    print(segment)

{'segment_id': 0, 'contig_id': '409_P1', 'genome_id': None, 'sequence_id': 0, 'absolute_coordinate': (0, 2000), 'relative_coordinate': (0, 2000), 'orientation': 'forward', 'label': 1, 'sequence': 'TACAAAACGATTAAAGGAGGTGGTAAAACATGACGCAGATGGACTTAGAGAAGTTCCTCTTGATGCTGGATACTGAACGTGCTAGACTCATGCTGCAAGACCTGCGGGATGACTCTAAGCGTTCACCTCAGCTTTACAACGCCATTGAGAAGCTACTGGCTCGTCACAACTTCGTGCTAAGCAAGGTGTCTGTTGACGAGAAGACGCTCGCTGATATGGAGGCTTTGAACAGGGAGTACGATAAGGTGCTCTCAGCTACTGAGGATAACGACACCGGGTATGGTGTACAGTAAGTGTTAGACTCAAGGTCATTACTATATGTAGTGGCCTTTATGGTTAACACACACTACGGGCGGAGACGCTACGTGGATTCTGAGAAATGAGGAGGGCAACTATGCTCGAACTTTTATGTAAAGCAATCCCGTGGCTGGTTGCAGGACTCCTGTTCTGTTCTGGCTACTGGGTGGCAGACAATAAGTGGGAGGCGAAGGTAAACAATGAGTACATCACGAAACTTGAGGCACGAGAGAATCAACGGGCTGCTGTCCAAGGTGAAATCAACAAAGTGTCAGCCGAGTGGCAAGACAAGATGTCCGCGCTGGAAGGCTCTACTGATAGGATTATTGCTGACCTTAACCGTGATAACAAGCGGCTGCGCGTCCGAGTCAACACCTCAGGTCTCACCGAGTCAGATGTCAGTAGATGCTTCCCTAATGGTAGAGTCGAACTACACCCAGAAACTTCTAAAAGTCTTATCCGATTAACTCAGGAGGCAGACCTTAAGGAGAAGGCGCTTCAGGACA

### Embedding Segment Dataset

In [43]:
train_dataset = RandomSegmentDataset(
    sequence_dataset=train_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    embedding_dataset=emb_ds_train,
    return_sequence=False,
    return_embedding=True,
    coverage=2,
    seed=42,
)
test_dataset = ContiguousSegmentDataset(
    sequence_dataset=test_seq_ds,
    min_segment_length=min_segment_length,
    max_segment_length=max_segment_length,
    embedding_dataset=emb_ds_test,
    return_sequence=False,
    return_embedding=True
)

In [44]:
for i, segment in enumerate(iter(train_dataset)):
    if i >= 3: break
    print(segment)

{'segment_id': 52833, 'contig_id': 'PHAA0000796', 'genome_id': None, 'sequence_id': 821, 'absolute_coordinate': (53526916, 53528916), 'relative_coordinate': (1743, 3743), 'orientation': 'forward', 'label': 0, 'embedding': tensor([[-0.4334,  0.9778, -0.1695,  ..., -0.1734, -0.3486,  1.1079],
        [-0.6050,  0.7644,  0.0928,  ..., -0.1520, -0.6110,  1.3234],
        [-0.4916,  1.0450, -0.0961,  ..., -0.2119, -0.4322,  1.3579],
        ...,
        [-0.4858,  0.9863, -0.1468,  ..., -0.1735, -0.3995,  1.2416],
        [-0.4288,  1.0664, -0.1840,  ..., -0.0914, -0.3371,  1.4859],
        [-0.5051,  0.9331, -0.2189,  ..., -0.3900, -0.5703,  1.4748]])}
{'segment_id': 476, 'contig_id': 'PHAA0000775', 'genome_id': None, 'sequence_id': 8, 'absolute_coordinate': (506240, 508240), 'relative_coordinate': (5700, 7700), 'orientation': 'forward', 'label': 0, 'embedding': tensor([[-0.5453,  1.0621, -0.1093,  ..., -0.2175, -0.5182,  1.1205],
        [-0.4964,  0.8034,  0.0159,  ..., -0.2271, -0.3527,

In [47]:
# validate shuffling
from itertools import islice
for shuffle in (False, True):
    train_dataset.shuffle = shuffle
    labels = [seg["label"] for seg in islice(train_dataset, 300)]
    print(f"shuffle={shuffle}: {''.join(str(l) for l in labels[:100])}")

shuffle=False: 0000000000000000000000000000000000000000000000000000000000011111111111111111111111111111111111111111
shuffle=True: 0000000100101001110010000001101001100101100111011000111111001001110101001111110111111111110111011011


### Data collator

In [48]:
tokenizer = LCATokenizer(kmer = kmer, shift = shift)
collator = SegmentDataCollator(
    tokenizer=tokenizer,
    add_special_tokens=add_special_tokens,
    return_metadata=return_metadata,
    has_embedding=True, # comment out if not using embeddings (or set to false)
)
loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collator)

In [51]:
# quick check of batch shapes
batch = next(iter(loader))

# using sequences
# print(batch["input_ids"].shape)
# print(batch["attention_mask"].shape)

# using embeddings
print(batch["embeddings"].shape)
print(batch["embedding_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 11, 384])
torch.Size([16, 11])
torch.Size([16])


## Model training

In [53]:
model_name = "neuralbioinfo/prokbert-mini-long"
# num_labels = len(train_ds.unique("label")) # 2

In [ ]:
# using sequences
# from prokbert.models import ProkBertForSequenceClassification
# from prokbert.models.megatronbert import ProkBertForSequenceClassification
# model = ProkBertForSequenceClassification.from_pretrained(model_name, trust_remote_code=True)

In [54]:
# using embeddings
class SegmentEmbeddingClassifier(nn.Module):

    def __init__(self, hidden_size: int, num_labels: int, dropout: float = 0.1):
        super().__init__()

        self.weighting_layer = nn.Linear(hidden_size, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

        # neutral init: uniform weights over the real rows at the start (= masked mean)
        nn.init.zeros_(self.weighting_layer.weight)
        nn.init.zeros_(self.weighting_layer.bias)

        # comment out if you do not want mean pooling
        for p in self.weighting_layer.parameters():
            p.requires_grad_(False)

    def forward(self, embeddings, embedding_mask, labels=None):
        keep = embedding_mask.bool().unsqueeze(-1) # (bs, rows, 1)
        assert keep.any(dim=1).all(), "every segment needs at least one real row"

        scores = self.weighting_layer(embeddings)  # (bs, rows, 1)
        # padded segment should have 0 weight
        scores = scores.masked_fill(~keep, float("-inf"))
        # sums to 1 over real rows
        weights = torch.softmax(scores.float(), dim=1).to(embeddings.dtype) # (bs, rows, 1)
        # (bs, rows, 1) * (bs, rows, hidden) -> (bs, hidden)
        pooled = (weights * embeddings).sum(dim=1)
        #  (bs, hidden) -> (bs, num_labels)
        logits = self.classifier(self.dropout(pooled))
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

model = SegmentEmbeddingClassifier(hidden_size=384, num_labels=2)

In [56]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({trainable / total:.2%})")

# full model
# trainable: 26,433,027 / 26,433,027 (100.00%)
# head only (with weighting layer)
# trainable: 1,155 / 1,155 (100.00%)
# head only (with mean pooling)
# trainable: 1,155 / 1,155 (100.00%)

trainable: 770 / 1,155 (66.67%)


In [58]:
# quick check
# model(**batch)

In [59]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):   # some models also return hidden states
        logits = logits[0]
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
        "mcc": matthews_corrcoef(labels, preds),
    }

In [61]:
training_args = TrainingArguments(
    output_dir=f"{base_dir}/phastyle-sequence-db-v2-embedding-db-mini-long-head",
    max_steps=20_000,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 4,
    learning_rate=5e-5,
    warmup_steps=0.1,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="balanced_accuracy",
    dataloader_num_workers=4,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to="wandb",
    run_name="phastyle-sequence-db-v2-embedding-db-mini-long-head",
    seed=42,
)

In [62]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Run training as script:


```bash
srun --job-name=phastyle-sequence-db-embedding-db-mini-long-head \
     --nodes=1 \
     --nodelist=bolka \
     --ntasks-per-node=1 \
     --cpus-per-task=8 \
     --gres=gpu:1 \
     --mem=150G \
     --time=7-00:00:00 \
     --partition=hi \
     --output=/home/osvathm/prokbert/logs_training/%x_%j.out \
     --error=/home/osvathm/prokbert/logs_training/%x_%j.err \
     torchrun --standalone --nproc_per_node=1 examples/phastyle_training.py
```